In [22]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/assessment_clean.csv")
df = df.sort_values(['Email', 'assignment_order']).reset_index(drop=True)

print(f"Loaded: {df.shape}")
print(f"Students: {df['Email'].nunique()}")
print(f"Assignments: {df['Assignment_Name'].nunique()}")
print()
print("Columns:")
print(df.columns.tolist())

Loaded: (840, 10)
Students: 42
Assignments: 20

Columns:
['Email', 'Assignment_Name', 'Total_Points_Earned', 'Total_Points_Possible', 'Time_Spent', 'source_file', 'grade_pct', 'time_minutes', 'never_started', 'assignment_order']


In [23]:

# Check which assignments have very high skip rates
skip_analysis = df.groupby('Assignment_Name').agg(
    skip_rate=('never_started', 'mean'),
    mean_grade=('grade_pct', 'mean'),
    student_count=('Email', 'count')
).round(3)

print("Assignments with high skip rates:")
print(skip_analysis[skip_analysis['skip_rate'] > 0.8].sort_values(
    'skip_rate', ascending=False
).to_string())

Assignments with high skip rates:
                                                                    skip_rate  mean_grade  student_count
Assignment_Name                                                                                         
IT207 - Lab8 - RESTfulWebService                                        1.000         0.0             42
IT207-Assignment6 - BorrowEase: A Node.js & MySQL Library Solution      1.000         0.0             42
IT207-Lab11-MySQL Stored Procedures                                     1.000         0.0             42
IT207-Lab12-Converting the NodejsTerminal App into a REST API           1.000         0.0             42
Midterm Mock Exam Review                                                0.905         0.0             42


In [24]:
# Remove incomplete/abandoned assignments
# These have 100% skip rate and 0% mean grade — they skew everything

EXCLUDE_ASSIGNMENTS = [
    'IT207 - Lab8 - RESTfulWebService',
    'IT207-Assignment6 - BorrowEase: A Node.js & MySQL Library Solution',
    'IT207-Lab11-MySQL Stored Procedures',
    'IT207-Lab12-Converting the NodejsTerminal App into a REST API',
    'Midterm Mock Exam Review',
]

before = len(df)
df = df[~df['Assignment_Name'].isin(EXCLUDE_ASSIGNMENTS)].copy()
after = len(df)

print(f"Removed {before - after} rows ({len(EXCLUDE_ASSIGNMENTS)} assignments)")
print(f"Remaining rows: {after}")
print(f"Remaining assignments: {df['Assignment_Name'].nunique()}")
print()
print("Assignments kept:")
for a in sorted(df['Assignment_Name'].unique()):
    print(f"  {a}")

Removed 210 rows (5 assignments)
Remaining rows: 630
Remaining assignments: 15

Assignments kept:
  IT207 - Lab8 - RESTfulWebService_New
  IT207-Assignment1- Displaying the User Agent
  IT207-Assignment2- Switching to Async APIs
  IT207-Assignment3 - TODOServer
  IT207-Assignment4 - Using the WHATWG API
  IT207-Assignment5- Chilly Delights REST API Server
  IT207-Lab0-ProgrammingReview
  IT207-Lab1-TheBigPicture
  IT207-Lab10-AccessingMySQLServerFromNodejs
  IT207-Lab2-SynchronousFileAPIs
  IT207-Lab3-AsynchronousFilesystemAPI
  IT207-Lab4-Events
  IT207-Lab5-ServerFundamentals
  IT207-Lab7-URL_QueryString
  IT207-Lab9-SQL_Review


In [25]:
# Remap assignment_order to 0-14 for the 15 remaining assignments
# Correct course order based on curriculum sequence

COURSE_ORDER_NEW = {
    'IT207-Lab0-ProgrammingReview':                          0,
    'IT207-Lab1-TheBigPicture':                              1,
    'IT207-Lab2-SynchronousFileAPIs':                        2,
    'IT207-Lab3-AsynchronousFilesystemAPI':                  3,
    'IT207-Lab4-Events':                                     4,
    'IT207-Lab5-ServerFundamentals':                         5,
    'IT207-Assignment1- Displaying the User Agent':          6,
    'IT207-Assignment2- Switching to Async APIs':            7,
    'IT207-Assignment3 - TODOServer':                        8,
    'IT207-Lab7-URL_QueryString':                            9,
    'IT207-Assignment4 - Using the WHATWG API':             10,
    'IT207 - Lab8 - RESTfulWebService_New':                 11,
    'IT207-Assignment5- Chilly Delights REST API Server':   12,
    'IT207-Lab9-SQL_Review':                                13,
    'IT207-Lab10-AccessingMySQLServerFromNodejs':           14,
}

df['assignment_order'] = df['Assignment_Name'].map(COURSE_ORDER_NEW)

# Check for any unmapped assignments
unmapped = df[df['assignment_order'].isna()]['Assignment_Name'].unique()
if len(unmapped) > 0:
    print("WARNING — unmapped assignments:")
    for a in unmapped:
        print(f"  '{a}'")
else:
    print("All 15 assignments mapped to new order 0-14")

print()
print("Assignment order verification:")
order_check = df[['Assignment_Name', 'assignment_order']].drop_duplicates()
order_check = order_check.sort_values('assignment_order')
for _, row in order_check.iterrows():
    print(f"  {int(row['assignment_order']):2d}. {row['Assignment_Name']}")

All 15 assignments mapped to new order 0-14

Assignment order verification:
   0. IT207-Lab0-ProgrammingReview
   1. IT207-Lab1-TheBigPicture
   2. IT207-Lab2-SynchronousFileAPIs
   3. IT207-Lab3-AsynchronousFilesystemAPI
   4. IT207-Lab4-Events
   5. IT207-Lab5-ServerFundamentals
   6. IT207-Assignment1- Displaying the User Agent
   7. IT207-Assignment2- Switching to Async APIs
   8. IT207-Assignment3 - TODOServer
   9. IT207-Lab7-URL_QueryString
  10. IT207-Assignment4 - Using the WHATWG API
  11. IT207 - Lab8 - RESTfulWebService_New
  12. IT207-Assignment5- Chilly Delights REST API Server
  13. IT207-Lab9-SQL_Review
  14. IT207-Lab10-AccessingMySQLServerFromNodejs


In [26]:
assignment_stats = df.groupby('Assignment_Name').agg(
    class_mean_grade=('grade_pct', 'mean'),
    class_std_grade=('grade_pct', 'std'),
    class_mean_time=('time_minutes', 'mean'),
    class_std_time=('time_minutes', 'std'),
    class_skip_rate=('never_started', 'mean')
).reset_index()

print("Class statistics per assignment:")
print(assignment_stats[['Assignment_Name',
                          'class_mean_grade',
                          'class_skip_rate']].to_string(index=False))

Class statistics per assignment:
                                   Assignment_Name  class_mean_grade  class_skip_rate
              IT207 - Lab8 - RESTfulWebService_New         80.930476         0.095238
      IT207-Assignment1- Displaying the User Agent         80.793333         0.071429
        IT207-Assignment2- Switching to Async APIs         86.444524         0.071429
                    IT207-Assignment3 - TODOServer         63.237381         0.095238
          IT207-Assignment4 - Using the WHATWG API         84.031429         0.095238
IT207-Assignment5- Chilly Delights REST API Server         18.008095         0.547619
                      IT207-Lab0-ProgrammingReview         74.857143         0.071429
                          IT207-Lab1-TheBigPicture         79.595238         0.119048
        IT207-Lab10-AccessingMySQLServerFromNodejs         15.452381         0.666667
                    IT207-Lab2-SynchronousFileAPIs         84.952381         0.095238
              IT207-L

In [27]:
df = df.merge(assignment_stats, on='Assignment_Name', how='left')

df['grade_zscore'] = (
    (df['grade_pct'] - df['class_mean_grade']) /
    (df['class_std_grade'] + 0.001)
).round(4)

df['time_zscore'] = (
    (df['time_minutes'] - df['class_mean_time']) /
    (df['class_std_time'] + 0.001)
).round(4)

df['effort_efficiency'] = (
    df['grade_pct'] / (df['time_minutes'] + 1)
).round(4)

print("Cohort-relative features added:")
print(df[['Email', 'Assignment_Name', 'grade_pct',
          'grade_zscore', 'time_zscore',
          'effort_efficiency']].head(10).to_string(index=False))

Cohort-relative features added:
           Email                              Assignment_Name  grade_pct  grade_zscore  time_zscore  effort_efficiency
afarooq8@gmu.edu                 IT207-Lab0-ProgrammingReview       84.0        0.2534      -0.5764             1.2999
afarooq8@gmu.edu                     IT207-Lab1-TheBigPicture      100.0        0.5652      -0.7588             3.5311
afarooq8@gmu.edu               IT207-Lab2-SynchronousFileAPIs      100.0        0.4848      -0.1544             1.3793
afarooq8@gmu.edu         IT207-Lab3-AsynchronousFilesystemAPI       15.0       -2.1063      -1.4034             2.8302
afarooq8@gmu.edu                            IT207-Lab4-Events       98.0        0.5257      -0.4664             1.3502
afarooq8@gmu.edu                IT207-Lab5-ServerFundamentals      100.0        0.4979      -0.1169             1.0562
afarooq8@gmu.edu IT207-Assignment1- Displaying the User Agent      100.0        0.5084      -0.4470             0.7288
afarooq8@gmu.edu

In [28]:
df = df.sort_values(['Email', 'assignment_order']).reset_index(drop=True)

df['grade_velocity'] = df.groupby('Email')['grade_pct'].diff()
df['time_velocity']  = df.groupby('Email')['time_minutes'].diff()

df['rolling_avg_grade_3'] = (
    df.groupby('Email')['grade_pct']
    .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    .round(2)
)

df['cumulative_skips'] = (
    df.groupby('Email')['never_started']
    .cumsum()
)

df['cumulative_avg_grade'] = (
    df.groupby('Email')['grade_pct']
    .transform(lambda x: x.expanding().mean())
    .round(2)
)

print("Sequential features added:")
print(df[['Email', 'assignment_order', 'grade_pct',
          'grade_velocity', 'rolling_avg_grade_3',
          'cumulative_skips']].head(25).to_string(index=False))

Sequential features added:
           Email  assignment_order  grade_pct  grade_velocity  rolling_avg_grade_3  cumulative_skips
afarooq8@gmu.edu                 0       84.0             NaN                84.00                 0
afarooq8@gmu.edu                 1      100.0            16.0                92.00                 0
afarooq8@gmu.edu                 2      100.0             0.0                94.67                 0
afarooq8@gmu.edu                 3       15.0           -85.0                71.67                 0
afarooq8@gmu.edu                 4       98.0            83.0                71.00                 0
afarooq8@gmu.edu                 5      100.0             2.0                71.00                 0
afarooq8@gmu.edu                 6      100.0             0.0                99.33                 0
afarooq8@gmu.edu                 7        0.0          -100.0                66.67                 0
afarooq8@gmu.edu                 8      100.0           100.0   

In [29]:
df['grade_velocity'] = df['grade_velocity'].fillna(0)
df['time_velocity']  = df['time_velocity'].fillna(0)

print("NaN check after filling:")
print(df[['grade_velocity', 'time_velocity',
          'rolling_avg_grade_3', 'cumulative_skips',
          'cumulative_avg_grade']].isnull().sum())

NaN check after filling:
grade_velocity          0
time_velocity           0
rolling_avg_grade_3     0
cumulative_skips        0
cumulative_avg_grade    0
dtype: int64


In [30]:
max_time = assignment_stats['class_mean_time'].max()
assignment_stats['time_ratio'] = (
    assignment_stats['class_mean_time'] / (max_time + 0.001)
)

assignment_stats['difficulty_score'] = (
    (1 - assignment_stats['class_mean_grade'] / 100) * 0.50 +
    assignment_stats['class_skip_rate'] * 0.30 +
    assignment_stats['time_ratio'] * 0.20
).round(4)

min_d = assignment_stats['difficulty_score'].min()
max_d = assignment_stats['difficulty_score'].max()
assignment_stats['difficulty_score'] = (
    (assignment_stats['difficulty_score'] - min_d) /
    (max_d - min_d + 0.001) * 100
).round(2)

df = df.merge(
    assignment_stats[['Assignment_Name', 'difficulty_score']],
    on='Assignment_Name',
    how='left'
)

difficulty_ranking = assignment_stats[['Assignment_Name',
                                        'difficulty_score',
                                        'class_mean_grade',
                                        'class_skip_rate']].sort_values(
    'difficulty_score', ascending=False
)
print("Assignment difficulty ranking (hardest to easiest):\n")
print(difficulty_ranking.to_string(index=False))

Assignment difficulty ranking (hardest to easiest):

                                   Assignment_Name  difficulty_score  class_mean_grade  class_skip_rate
        IT207-Lab10-AccessingMySQLServerFromNodejs             99.81         15.452381         0.666667
IT207-Assignment5- Chilly Delights REST API Server             98.66         18.008095         0.547619
                    IT207-Assignment3 - TODOServer             47.36         63.237381         0.095238
      IT207-Assignment1- Displaying the User Agent             37.88         80.793333         0.071429
                      IT207-Lab0-ProgrammingReview             23.10         74.857143         0.071429
              IT207 - Lab8 - RESTfulWebService_New             21.44         80.930476         0.095238
                             IT207-Lab9-SQL_Review             20.99         75.857143         0.095238
          IT207-Assignment4 - Using the WHATWG API             16.48         84.031429         0.095238
           

In [31]:
df['next_grade'] = df.groupby('Email')['grade_pct'].shift(-1)

median_grade = df.groupby('Email')['grade_pct'].mean().median()
print(f"Median final average grade: {median_grade:.1f}%")

final_avg = df.groupby('Email')['grade_pct'].mean().rename('final_avg_grade')
df = df.merge(final_avg, on='Email', how='left')
df['at_risk'] = (df['final_avg_grade'] < median_grade).astype(int)

at_risk_students = df.groupby('Email')['at_risk'].first()
print("At-risk distribution:")
print(at_risk_students.value_counts())
print(f"\n{at_risk_students.mean()*100:.1f}% of students labeled at-risk")
print(f"Threshold used: {median_grade:.1f}%")

Median final average grade: 81.4%
At-risk distribution:
at_risk
1    21
0    21
Name: count, dtype: int64

50.0% of students labeled at-risk
Threshold used: 81.4%


In [32]:
MODEL_FEATURES = [
    'Email',
    'Assignment_Name',
    'assignment_order',
    'grade_pct',
    'time_minutes',
    'never_started',
    'grade_zscore',
    'time_zscore',
    'effort_efficiency',
    'class_skip_rate',
    'grade_velocity',
    'time_velocity',
    'rolling_avg_grade_3',
    'cumulative_skips',
    'cumulative_avg_grade',
    'difficulty_score',
    'next_grade',
    'at_risk',
]

df_features = df[MODEL_FEATURES].copy()
df_features = df_features[df_features['next_grade'].notna()].copy()

print(f"Feature matrix shape: {df_features.shape}")
print(f"Input features: 13")
print(f"NaN check: {df_features.isnull().sum().sum()} missing values")

df_features.to_csv("../data/processed/features.csv", index=False)
print("\nSaved: data/processed/features.csv")
print("Phase 2 complete.")

Feature matrix shape: (588, 18)
Input features: 13
NaN check: 0 missing values

Saved: data/processed/features.csv
Phase 2 complete.
